# Craft My Book

Uses the `llm` package in `src/` (OpenAI provider) to draft book content.

In [1]:
import sys
from pathlib import Path


sys.path.insert(0, str(Path.cwd() / "src"))

from llm import get_client, chat
import config

In [ ]:
# Generic demo of the raw `llm` package - not tied to any pipeline component/config,
# just a standalone example of calling a model directly.
client = get_client("openai")
model = "gpt-4o-mini"

In [3]:
prompt = "Write an engaging opening paragraph for a book about ..."

response = chat(client, model, prompt)
print(response)

Certainly! What specific theme or subject would you like the opening paragraph to focus on?


## Module 1.2 — Speech Processing (Whisper)

Turns a lecture recording — video or audio — into a structured, timestamped,
domain-aware transcript (`src/ingestion/speech.py`, `src/ingestion/vocab.py`).
Requires the `ffmpeg` binary on PATH and `faster-whisper` installed
(`pip install -r requirements.txt`).

**Design**: `ingestion.base.Ingestor` is an abstract strategy — `ingest(source_path) ->
IngestedDocument` — that every source type implements (speech today; PDF/DOCX/PPTX/
image strategies for Pipeline A land the same way later). `SpeechIngestor` is the
speech strategy; `ingestion.registry.get_ingestor_class()` picks the right strategy
for a file by extension, so code that walks a folder of mixed sources never branches
on file type itself. `IngestedDocument`/`IngestedSegment` are the common output shape
every strategy produces, regardless of what ran underneath.

**Config**: `src/config/config.yaml` is nested by component, not one global
provider/model. `ingestion.speech` alone has three independently tunable model
choices — `whisper` (transcription), `cleaning_llm` (post-transcription cleanup),
`vocab_llm` (domain vocab extraction) — so e.g. cleaning can run on a cheap model
while vocab extraction runs on something else, without either affecting the other
or any future component (toc/writer). `src/config/__init__.py` loads this into typed
dataclasses; everything reads from `config.INGESTION.speech.*`, nothing is hardcoded
in a function signature. Whisper defaults to `"small"` for local iteration — bump
`ingestion.speech.whisper.model_size` to `"large-v3"` in the YAML for real
lecture-quality transcription once you have a stable connection (and ideally a GPU).

Flow: `extract_vocab` (LLM pass over slide titles/filenames/headings — vocab is
per-corpus, never hardcoded) → `extract_audio` (ffmpeg; handles video or audio
sources) → `transcribe_audio` (faster-whisper, vocab-primed, VAD-filtered, word
timestamps, no cross-segment conditioning) → `clean_transcript` (conservative LLM
pass — fixes terms/punctuation, changes nothing else) → `IngestedDocument` JSON.

In [ ]:
from ingestion import extract_audio, extract_vocab, vocab_material_from_filenames, bootstrap_vocab_from_audio

# Each sub-component gets its own client/model, independently tunable via
# config.INGESTION.speech.* - vocab extraction doesn't have to run on the same
# provider/model as transcript cleaning.
vocab_client = get_client(config.INGESTION.speech.vocab_llm.provider)
vocab_model = config.INGESTION.speech.vocab_llm.model

# data/harvard-speech.wav: ~34s of real spoken English (public-domain Harvard sentences
# test corpus) - use this to smoke-test the pipeline locally before pointing it at a
# real lecture.
source_path = "data/harvard-speech.wav"

# Preferred: derive vocab from material that already exists around the recording
# (slide titles / filenames here; could also be PDF headings).
sibling_files: list[str] = []  # no slides for this sample -> falls through to bootstrap
vocab_material = vocab_material_from_filenames(sibling_files)
vocab = extract_vocab(vocab_material, vocab_client, vocab_model)

# Fallback: audio is the ONLY source (no slides/useful filenames) - bootstrap vocab from
# a fast draft transcription pass instead of skipping priming altogether.
if not vocab:
    audio_path = extract_audio(source_path)
    vocab = bootstrap_vocab_from_audio(audio_path, vocab_client, vocab_model)

vocab

In [ ]:
# End to end via the strategy: registry picks the Ingestor for this file, ingest()
# runs source -> audio -> transcribe -> clean, save_document() writes the JSON. Once
# PDF/DOCX/image strategies exist, code that walks a mixed folder still looks like this.
from ingestion import get_ingestor_class, save_document

cleaning_client = get_client(config.INGESTION.speech.cleaning_llm.provider)
cleaning_model = config.INGESTION.speech.cleaning_llm.model

IngestorClass = get_ingestor_class(source_path)  # -> SpeechIngestor, by extension
ingestor = IngestorClass(client=cleaning_client, clean_model=cleaning_model, vocab=vocab)

document = ingestor.ingest(source_path)
out_path = save_document(document, config.INGESTION.output_dir)

print(f"{document.source_type}: {len(document.segments)} segments, {document.metadata['duration']:.0f}s -> {out_path}")
document.segments[0]